# Numerical propagation with propygator

This notebook walks through Feature 1.1: propagate an initial state vector with a
configurable force model, inspect the resulting `Trajectory`, and produce the
standard plot + CSV outputs.

Importing `propygator` does **not** start the JVM (`docs/architecture.md` §10);
the JVM spins up lazily on the first Orekit-touching call (here, the first
propagation).

In [ ]:
from pathlib import Path

import numpy as np

import propygator as pgr

pgr.__version__

## 1. Build an initial state

A `State` is a Cartesian position + velocity at an `Epoch`, in an explicit `Frame`.
Everything is SI (metres, m/s) and the frame must be inertial for propagation
(`EME2000` / `J2000`). Here we build a circular, ISS-like LEO at ~420 km altitude
and 51.6° inclination.

In [ ]:
MU = 3.986004418e14  # Earth GM (WGS84), m^3/s^2
r = 6798137.0  # ~420 km altitude
v = np.sqrt(MU / r)  # circular speed
inc = np.radians(51.6)

initial = pgr.State(
    pgr.Epoch.from_iso("2024-01-01T00:00:00"),
    np.array([r, 0.0, 0.0]),
    np.array([0.0, v * np.cos(inc), v * np.sin(inc)]),
    pgr.Frame.EME2000,
)
initial

## 2. Propagate

`propagate_numerical` returns a `Trajectory` in `EME2000`, sampled every
`output_step` seconds. With no `force_models` argument it uses the `leo_default`
preset (gravity field + Sun/Moon third body + drag + SRP + tides). `duration` and
`output_step` are in seconds.

In [ ]:
traj = pgr.propagate_numerical(initial, duration=86400, output_step=60)
len(traj), traj.frame

Every `Trajectory` carries a reproducibility record in `metadata` — versions,
integrator, tolerances, the exact list of forces that acted, and so on.

In [ ]:
dict(traj.metadata)

## 3. Inspect the trajectory

A `Trajectory` indexes and iterates as `State` objects, converts between frames in
bulk, interpolates to an arbitrary epoch with `at()`, and exposes osculating
Keplerian elements via `to_keplerian()`.

In [ ]:
first = traj[0]
kep = first.to_keplerian()
print(f"a = {kep.semi_major_axis_m / 1000:.1f} km")
print(f"e = {kep.eccentricity:.5f}")
print(f"i = {np.degrees(kep.inclination_rad):.2f} deg")

# Bulk frame conversion (EME2000 -> Earth-fixed ITRF).
itrf = traj.to_frame(pgr.Frame.ITRF)
itrf.frame

In [ ]:
# Hermite-interpolate to an epoch between samples (30 s after the start).
mid = pgr.Epoch.from_iso("2024-01-01T00:00:30")
traj.at(mid)

## 4. Configure the force model and spacecraft

`ForceModelConfig` ships three presets (`leo_default`, `geo_default`, `keplerian`)
and is fully customizable. `SpacecraftConfig` carries the mass and geometry; a
`box_and_panels` geometry drives both drag and SRP, and the attitude family
(`LofAligned`, `InPlaneTracking`, `LofOffset`, …) sets the orientation. A
`VariableCd` table gives a density-varying drag coefficient (`sphere_default()` is
shipped).

In [ ]:
box_traj = pgr.propagate_numerical(
    initial,
    duration=86400,
    output_step=60,
    force_models=pgr.ForceModelConfig.leo_default(),
    spacecraft=pgr.SpacecraftConfig(
        mass_kg=420,
        geometry=pgr.SpacecraftGeometry.box_and_panels(
            x_length_m=2.0,
            y_length_m=1.0,
            z_length_m=1.0,
            solar_array_area_m2=10.0,
            drag_coefficient=pgr.VariableCd.sphere_default(),
        ),
    ),
    attitude=pgr.InPlaneTracking(),
)
len(box_traj)

## 5. Plot

Each `plot_*` returns its native figure (matplotlib for 2-D, Plotly for the
interactive 3-D view), so you can post-process with the native API. `plot_summary`
is the default stacked view: ground track, altitude, and a speed panel per frame.

In [ ]:
pgr.plot_summary(traj)

In [ ]:
pgr.plot_ground_track(traj)

In [ ]:
# Interactive 3-D (Plotly). Use frame=pgr.Frame.ITRF with show_map_overlay=True
# to drape the coastline on a static Earth.
pgr.plot_3d(traj).show()

## 6. Export everything

`export_all` bundles the common case into an output directory: a summary PNG, an
interactive 3-D HTML, and a CSV (16 default columns plus a metadata header). It
returns a dict mapping each output name to the file written. Pass several
`frames_3d` to get one frame-suffixed 3-D file each, or use `export_csv` /
`plot_*` individually for finer control.

In [ ]:
outputs = pgr.export_all(
    traj,
    output_dir=Path("./run_01"),
    speed_frames=[pgr.Frame.EME2000, pgr.Frame.ITRF],
    frames_3d=[pgr.Frame.EME2000, pgr.Frame.ITRF],
)
outputs

---

That's the full Feature 1.1 loop: state → `propagate_numerical` → `Trajectory`
→ inspect / plot / export. The full contract (every preset, validation rule, and
output column) lives in `docs/features.md` §1.1.